In [2]:
import json
import os
import tempfile
import time
from pathlib import Path
from osgeo import gdal
import matplotlib.pyplot as plt
import pandas as pd
import pystac_client
import rasterio
import seaborn as sns

os.environ['GDAL_HTTP_MAX_RETRY'] = '5'
os.environ['GDAL_HTTP_RETRY_DELAY'] = '3'

STAC_URL = "https://data.inpe.br/bdc/stac/v1/"
COLLECTION_ID = 'CB4A-WFI-L4-DN-1'
SCENE_IDS = [
    "CBERS_4A_WFI_20260930_242_132_L4",
    "CBERS_4A_WFI_20260930_242_116_L4",
    "CBERS_4A_WFI_20260929_236_116_L4",
]
BANDS = ['BAND13','BAND14','BAND15','BAND16']

# Algoritmos de compressão para teste
GTIFF_COMPRESSIONS = ['deflate', 'lzw', 'zstd']
COG_COMPRESSIONS = ['deflate', 'lzw', 'zstd']

# Diretórios apenas para métricas e relatórios
OUTPUT_DIR = Path('/home/jovyan/Downloads/stac_CB4A-WFI-L4-DN-1')
METRICS_DIR = OUTPUT_DIR / 'metricas_json'
# ==============================================================================


def garante_dir(path: Path):
    """Garante que o diretório exista."""
    path.mkdir(parents=True, exist_ok=True)


# ==============================================================================
# MUDANÇA 2: FUNÇÃO DE BUSCA STAC COM RETENTATIVA E EXPONENTIAL BACKOFF
# ==============================================================================
def stac_search_with_retry(client, collection_id: str, scene_id: str, max_retries: int = 5) -> list:
    """Realiza a busca STAC com retentativas e espera exponencial em caso de erro 429."""
    for attempt in range(max_retries):
        try:
            search = client.search(collections=[collection_id], ids=[scene_id])
            return list(search.items())
        except Exception as e:
            if "429" in str(e) or "Too Many Requests" in str(e):
                wait_time = (2 ** attempt) + 3  # Espera incremental: 4s, 5s, 7s, 11s, 19s...
                print(f"   [RATE LIMIT 429] Limite atingido. Aguardando {wait_time}s antes de tentar novamente (Tentativa {attempt + 1}/{max_retries})...")
                time.sleep(wait_time)
            else:
                raise e
    raise Exception(f"Falha ao buscar a cena {scene_id} após {max_retries} tentativas devido ao Rate Limit.")


def evaluate_asset_in_temp(asset_href: str) -> list[dict]:
    """
    Lê o asset remoto, realiza os testes de compressão (GTiff e COG)
    em diretório temporário efêmero e retorna as métricas.
    Nenhum arquivo raster é salvo no diretório final.
    """
    results = []

    with tempfile.TemporaryDirectory() as temp_dir_str:
        temp_dir = Path(temp_dir_str)
        raw_path = temp_dir / "raw.tif"

        # 1. Leitura remota e gravação do RAW temporário
        with rasterio.open(asset_href) as src:
            profile = src.profile.copy()
            profile.update({
                'driver': 'GTiff',
                'compress': None,
                'tiled': True,
                'blockxsize': 256,
                'blockysize': 256,
            })
            data = src.read()

            with rasterio.open(raw_path, 'w', **profile) as dst:
                dst.write(data)

        raw_size_bytes = raw_path.stat().st_size
        raw_size_mb = raw_size_bytes / (1024 * 1024)

        t0_read = time.time()
        with rasterio.open(raw_path) as src:
            _ = src.read()
        raw_read_time = time.time() - t0_read

        # Linha de base RAW
        results.append({
            'Algoritmo': 'Sem Comp',
            'Tamanho_Final_MB': round(raw_size_mb, 2),
            'Taxa_Compressao (%)': 0.0,
            'Tempo_Compressao_segundos': 0.0,
            'Tempo_Leitura_segundos': round(raw_read_time, 3),
        })

        # 2. Testes em GeoTIFF convencional (DEFLATE, LZW, ZSTD)
        COMPRESSION_PROFILES = {
            'deflate': {'compress': 'deflate', 'zlevel': 1, 'predictor': 2},
            'lzw':     {'compress': 'lzw', 'predictor': 2},
            'zstd':    {'compress': 'zstd', 'zstd_level': 3, 'predictor': 2}
        }

        for comp in GTIFF_COMPRESSIONS:
            comp_path = temp_dir / f'gtiff_{comp}.tif'

            comp_opts = COMPRESSION_PROFILES[comp]
            profile.update({
                'tiled': True,
                'blockxsize': 256,
                'blockysize': 256,
                **comp_opts
            })

            start_write = time.time()
            with rasterio.open(comp_path, 'w', **profile) as dst:
                dst.write(data)
            time_write = time.time() - start_write

            start_read = time.time()
            with rasterio.open(comp_path) as src_comp:
                _ = src_comp.read()
            time_read = time.time() - start_read

            comp_size_bytes = comp_path.stat().st_size
            comp_size_mb = comp_size_bytes / (1024 * 1024)
            reduction = (1 - (comp_size_bytes / raw_size_bytes)) * 100

            results.append({
                'Algoritmo': comp.upper(),
                'Tamanho_Final_MB': round(comp_size_mb, 2),
                'Taxa_Compressao (%)': round(reduction, 2),
                'Tempo_Compressao_segundos': round(time_write, 3),
                'Tempo_Leitura_segundos': round(time_read, 3),
            })

        # 3. Testes em COG via GDAL (LZW, DEFLATE, ZSTD)
        COG_PROFILES = {
            'deflate': ["COMPRESS=DEFLATE", "LEVEL=9", "PREDICTOR=2", "NUM_THREADS=ALL_CPUS"],
            'lzw':     ["COMPRESS=LZW", "PREDICTOR=2", "NUM_THREADS=ALL_CPUS"],
            'zstd':    ["COMPRESS=ZSTD", "LEVEL=9", "PREDICTOR=2", "NUM_THREADS=ALL_CPUS"]
        }

        for cog_comp, cog_options in COG_PROFILES.items():
            cog_path = temp_dir / f'cog_{cog_comp}.tif'

            t0_cog = time.time()
            gdal.Translate(
                str(cog_path),
                str(raw_path),
                format="COG",
                creationOptions=cog_options
            )
            time_write_cog = time.time() - t0_cog

            start_read = time.time()
            with rasterio.open(cog_path) as src_cog:
                _ = src_cog.read()
            time_read_cog = time.time() - start_read

            cog_size_bytes = cog_path.stat().st_size
            cog_size_mb = cog_size_bytes / (1024 * 1024)
            reduction_cog = (1 - (cog_size_bytes / raw_size_bytes)) * 100

            results.append({
                'Algoritmo': f'COG_{cog_comp.upper()}',
                'Tamanho_Final_MB': round(cog_size_mb, 2),
                'Taxa_Compressao (%)': round(reduction_cog, 2),
                'Tempo_Compressao_segundos': round(time_write_cog, 3),
                'Tempo_Leitura_segundos': round(time_read_cog, 3),
            })

    return results


# ==============================================================================
# FASE 1: OBTENÇÃO DE MÉTRICAS E GRAVAÇÃO DE JSONs
# ==============================================================================
def executar_fase_processamento():
    """Baixa em memória temporária, calcula métricas e salva arquivos JSON."""
    garante_dir(OUTPUT_DIR)
    garante_dir(METRICS_DIR)

    print(f'Conectando ao STAC: {STAC_URL}...')
    client = pystac_client.Client.open(STAC_URL)

    for scene_id in SCENE_IDS:
        print('\n==================================================')
        print(f'Processando Cena: {scene_id}')
        print('==================================================')

        # Uso da função com retry para busca de cenas
        items = stac_search_with_retry(client, COLLECTION_ID, scene_id)

        if not items:
            print(f'Cena "{scene_id}" não encontrada na coleção.')
            continue

        item = items[0]

        for band_name in BANDS:
            json_metric_path = METRICS_DIR / f'metricas_{scene_id}_{band_name}.json'

            if json_metric_path.exists():
                print(f'   [CACHE] JSON já existe: {json_metric_path.name}')
                continue

            if band_name not in item.assets:
                print(f'Banda "{band_name}" não encontrada na cena {scene_id}.')
                continue

            print(f'\n--> Calculando métricas para Banda/Asset: {band_name}')
            asset_href = item.assets[band_name].href

            try:
                metrics = evaluate_asset_in_temp(asset_href)

                # Adiciona informações da cena e banda em cada registro do JSON
                for metric in metrics:
                    metric['Cena'] = scene_id
                    metric['Banda'] = band_name

                with open(json_metric_path, 'w', encoding='utf-8') as f:
                    json.dump(metrics, f, indent=4, ensure_ascii=False)
                print(f'   [MÉTRICAS SALVAS] {json_metric_path.name}')

            except Exception as e:
                print(f'   [ERRO AO PROCESSAR ASSET] {e}')

            # ==============================================================================
            # MUDANÇA 3: PAUSA PREVENTIVA ENTRE ASSETS PARA RESPEITAR LIMITE DA API
            # ==============================================================================
            time.sleep(2)


# ==============================================================================
# FASE 2: CARREGAMENTO DOS JSONs, GERAÇÃO DO CSV E GRÁFICOS
# ==============================================================================
def executar_fase_analise():
    """Carrega todos os arquivos JSON e gera a tabela CSV e os gráficos."""
    print('\n==================================================')
    print('   INICIANDO LEITURA DOS JSONs E GERAÇÃO DAS ANÁLISES')
    print('==================================================')

    json_files = list(METRICS_DIR.glob('*.json'))

    if not json_files:
        print(f'Nenhum arquivo JSON encontrado em {METRICS_DIR}.')
        return

    todas_metricas = []
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                dados = json.load(f)
                if isinstance(dados, list):
                    todas_metricas.extend(dados)
                elif isinstance(dados, dict):
                    todas_metricas.append(dados)
        except Exception as e:
            print(f'Erro ao ler arquivo {json_file.name}: {e}')

    if not todas_metricas:
        print('Nenhum dado válido extraído dos arquivos JSON.')
        return

    df = pd.DataFrame(todas_metricas)

    # 1. Exportação da Tabela CSV Consolidada
    csv_consolidado = OUTPUT_DIR / 'planilha_consolidada.csv'
    df.to_csv(csv_consolidado, index=False)
    print(f'Planilha CSV consolidada salva em: {csv_consolidado}')

    # 2. Geração do Painel Gráfico
    sns.set_theme(style='whitegrid')
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))

    # Gráfico 1: Tamanho Médio após Compressão
    ax1 = axes[0, 0]
    sns.barplot(
        data=df, x='Algoritmo', y='Tamanho_Final_MB', hue='Algoritmo',
        ax=ax1, palette='viridis', errorbar=None, legend=False
    )
    ax1.set_title('Tamanho Médio após Compressão (MB)')
    ax1.set_ylabel('Megabytes (MB)')
    ax1.tick_params(axis='x', rotation=30)
    for container in ax1.containers:
        ax1.bar_label(container, fmt='%.2f MB', padding=3)

    # Gráfico 2: Taxa Média da Compressão
    ax2 = axes[0, 1]
    sns.barplot(
        data=df, x='Algoritmo', y='Taxa_Compressao (%)', hue='Algoritmo',
        ax=ax2, palette='mako', errorbar=None, legend=False
    )
    ax2.set_title('Taxa Média de Compressão (%)')
    ax2.set_ylabel('Porcentagem (%)')
    ax2.tick_params(axis='x', rotation=30)
    for container in ax2.containers:
        ax2.bar_label(container, fmt='%.2f%%', padding=3)

    # Gráfico 3: Tempo Médio para Compressão
    ax3 = axes[1, 0]
    sns.barplot(
        data=df, x='Algoritmo', y='Tempo_Compressao_segundos', hue='Algoritmo',
        ax=ax3, palette='magma', errorbar=None, legend=False
    )
    ax3.set_title('Tempo Médio para Compressão (Segundos)')
    ax3.set_ylabel('Segundos (s)')
    ax3.tick_params(axis='x', rotation=30)
    for container in ax3.containers:
        ax3.bar_label(container, fmt='%.3fs', padding=3)

    # Gráfico 4: Tempo Médio para Leitura
    ax4 = axes[1, 1]
    sns.barplot(
        data=df, x='Algoritmo', y='Tempo_Leitura_segundos', hue='Algoritmo',
        ax=ax4, palette='crest', errorbar=None, legend=False
    )
    ax4.set_title('Tempo Médio para Leitura / Descompactação (Segundos)')
    ax4.set_ylabel('Segundos (s)')
    ax4.tick_params(axis='x', rotation=30)
    for container in ax4.containers:
        ax4.bar_label(container, fmt='%.3fs', padding=3)

    plt.tight_layout()
    grafico_path = OUTPUT_DIR / 'graficos_comparativos_reais.png'
    plt.savefig(grafico_path, dpi=150)
    plt.close()
    print(f'Gráficos comparativos salvos em: {grafico_path}')

    # Resumo descritivo no terminal
    df_resumo = (
        df.groupby('Algoritmo')
        .agg(
            Tamanho_Medio_MB=('Tamanho_Final_MB', 'mean'),
            Taxa_Media_Compressao=('Taxa_Compressao (%)', 'mean'),
            Tempo_Medio_Compressao_s=('Tempo_Compressao_segundos', 'mean'),
            Tempo_Medio_Leitura_s=('Tempo_Leitura_segundos', 'mean'),
        )
        .reset_index()
        .round({'Tamanho_Medio_MB': 2, 'Taxa_Media_Compressao': 2, 
                'Tempo_Medio_Compressao_s': 3, 'Tempo_Medio_Leitura_s': 3})
    )

    print('\n' + '=' * 20 + ' RESUMO DAS MÉDIAS ' + '=' * 20)
    print(df_resumo.to_string(index=False))
    print('=' * 65)


def main():
    executar_fase_processamento()
    executar_fase_analise()


if __name__ == '__main__':
    main()

Conectando ao STAC: https://data.inpe.br/bdc/stac/v1/...

Processando Cena: CBERS_4A_WFI_20260930_242_132_L4
   [CACHE] JSON já existe: metricas_CBERS_4A_WFI_20260930_242_132_L4_BAND13.json
   [CACHE] JSON já existe: metricas_CBERS_4A_WFI_20260930_242_132_L4_BAND14.json
   [CACHE] JSON já existe: metricas_CBERS_4A_WFI_20260930_242_132_L4_BAND15.json
   [CACHE] JSON já existe: metricas_CBERS_4A_WFI_20260930_242_132_L4_BAND16.json

Processando Cena: CBERS_4A_WFI_20260930_242_116_L4
   [CACHE] JSON já existe: metricas_CBERS_4A_WFI_20260930_242_116_L4_BAND13.json
   [CACHE] JSON já existe: metricas_CBERS_4A_WFI_20260930_242_116_L4_BAND14.json
   [CACHE] JSON já existe: metricas_CBERS_4A_WFI_20260930_242_116_L4_BAND15.json
   [CACHE] JSON já existe: metricas_CBERS_4A_WFI_20260930_242_116_L4_BAND16.json

Processando Cena: CBERS_4A_WFI_20260929_236_116_L4
   [CACHE] JSON já existe: metricas_CBERS_4A_WFI_20260929_236_116_L4_BAND13.json
   [CACHE] JSON já existe: metricas_CBERS_4A_WFI_20260929_